In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error
from torch.utils.data import Dataset, DataLoader, TensorDataset
import os

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
from torch.utils.data import Dataset, DataLoader, TensorDataset
import os
import time
import warnings
from tqdm import tqdm  # 用于显示进度条

# 忽略警告
warnings.filterwarnings('ignore')

# 配置参数
class Config:
    SEQ_LEN = 7 * 96  # 7天数据
    PRED_LEN = 12      # 预测长度（24个时间点）
    TEST_START_DAY = 15  # 开始测试的第几天
    TEST_END_DAY = 30    # 结束测试的第几天
    
    # LSTM模型参数
    HIDDEN_SIZE = 128
    NUM_LAYERS = 2
    DROPOUT = 0.2
    
    # 训练参数
    BATCH_SIZE = 128
    LEARNING_RATE = 0.0005
    NUM_EPOCHS = 100
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    PRINT_EVERY = 1  # 每个epoch都打印进度

# 自定义数据集类
class ElectricityPriceDataset(Dataset):
    def __init__(self, prices, seq_len, pred_len):
        self.prices = prices
        self.seq_len = seq_len
        self.pred_len = pred_len
        
        # 创建输入序列和目标序列
        self.X, self.y = self.create_sequences()
    
    def create_sequences(self):
        X, y = [], []
        total_points = len(self.prices)
        
        # 为每个可能的序列创建输入和目标
        if total_points < self.seq_len + self.pred_len:
            return np.array([]), np.array([])
        
        for i in range(total_points - self.seq_len - self.pred_len + 1):
            X.append(self.prices[i:i+self.seq_len])
            y.append(self.prices[i+self.seq_len:i+self.seq_len+self.pred_len])
        
        return np.array(X), np.array(y)
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return torch.FloatTensor(self.X[idx]), torch.FloatTensor(self.y[idx])

# LSTM预测模型
class PricePredictor(nn.Module):
    def __init__(self, input_size=1, hidden_size=Config.HIDDEN_SIZE, 
                 num_layers=Config.NUM_LAYERS, output_size=Config.PRED_LEN, 
                 dropout=Config.DROPOUT):
        super(PricePredictor, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        
        # LSTM层
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0
        )
        
        # 全连接层
        self.fc = nn.Sequential(
            nn.Linear(hidden_size, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, output_size)
        )
    
    def forward(self, x):
        # 初始化隐藏状态
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(Config.DEVICE)
        c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(Config.DEVICE)
        
        # 检查输入维度并调整为3D
        if x.dim() == 2:
            # 如果输入是2D (批次大小, 序列长度)，添加特征维度
            x = x.unsqueeze(-1)  # 变为 (批次大小, 序列长度, 1)
        elif x.dim() == 3:
            # 如果输入已经是3D，保持不变
            pass
        else:
            raise ValueError(f"输入维度 {x.dim()} 不支持，应为2D或3D")
        
        # LSTM前向传播
        out, _ = self.lstm(x, (h0, c0))
        
        # 只取最后一个时间步的输出
        out = out[:, -1, :]
        
        # 全连接层生成预测
        return self.fc(out)

# 主函数
def main():
    print(f"使用设备: {Config.DEVICE}")
    print(f"序列长度: {Config.SEQ_LEN} (相当于 {Config.SEQ_LEN // 96} 天)")
    print(f"预测长度: {Config.PRED_LEN} 个时间点 (相当于 {Config.PRED_LEN / 4} 小时)")
    
    # 1. 加载数据
    print("加载数据...")
    data_path = os.path.join('..', 'data', 'test_prices.csv')
    
    if not os.path.exists(data_path):
        print(f"错误: 文件不存在 - {data_path}")
        return
    
    data = pd.read_csv(data_path)
    print(f"数据加载成功，共 {len(data)} 行")
    
    # 提取电价列（假设最后一列是电价）
    if len(data.columns) < 1:
        print("错误: 数据文件没有列")
        return
        
    price_column = data.columns[-1]
    prices = data[price_column].values.astype(float)
    print(f"使用列 '{price_column}' 作为电价数据，共 {len(prices)} 个数据点")
    
    # 转换为一维数组
    prices = prices.reshape(-1, 1)
    
    # 2. 数据预处理 - 归一化
    scaler = MinMaxScaler(feature_range=(0, 1))
    scaled_prices = scaler.fit_transform(prices)
    
    # 3. 准备训练集和测试集
    total_days = len(prices) // 96  # 计算总天数
    print(f"总天数: {total_days}")
    
    # 检查是否有足够的数据
    if total_days < 14:
        print(f"错误: 只有 {total_days} 天的数据，至少需要 14 天数据")
        return
    
    # 训练集：前14天的数据
    train_data = scaled_prices[:14 * 96]
    print(f"训练集大小: {len(train_data)} 个点")
    
    # 测试集：第15-30天的数据
    test_start_idx = (Config.TEST_START_DAY - 1) * 96
    test_end_idx = min(Config.TEST_END_DAY * 96, len(scaled_prices))
    test_data = scaled_prices[test_start_idx:test_end_idx]
    test_days = (test_end_idx - test_start_idx) // 96
    print(f"测试集天数: {test_days} (第 {Config.TEST_START_DAY} 到 {Config.TEST_START_DAY + test_days - 1} 天)")
    
    # 4. 创建数据集和数据加载器
    train_dataset = ElectricityPriceDataset(
        train_data.flatten(),  # 使用flatten将二维数组转换为一维
        Config.SEQ_LEN, 
        Config.PRED_LEN
    )
    
    # 检查是否有足够的训练样本
    if len(train_dataset) == 0:
        print("错误: 训练数据集为空！")
        print(f"所需最小数据点数: {Config.SEQ_LEN + Config.PRED_LEN}")
        print(f"实际训练数据点数: {len(train_data)}")
        return
    
    print(f"创建了 {len(train_dataset)} 个训练样本")
    
    train_loader = DataLoader(
        train_dataset, 
        batch_size=Config.BATCH_SIZE, 
        shuffle=True
    )
    
    # 5. 初始化模型、损失函数和优化器
    model = PricePredictor().to(Config.DEVICE)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=Config.LEARNING_RATE)
    
    # 6. 训练模型 - 使用进度条显示训练进度
    print("\n开始训练模型...")
    train_losses = []
    start_time = time.time()
    
    # 使用tqdm创建进度条
    progress_bar = tqdm(range(Config.NUM_EPOCHS), desc="训练进度", unit="epoch")
    
    for epoch in progress_bar:
        model.train()
        epoch_loss = 0
        batch_count = 0
        
        # 使用tqdm显示批次进度
        batch_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{Config.NUM_EPOCHS}", leave=False)
        
        for inputs, targets in batch_bar:
            inputs = inputs.to(Config.DEVICE)
            targets = targets.to(Config.DEVICE)
            
            # 前向传播
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            
            # 反向传播和优化
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            epoch_loss += loss.item()
            batch_count += 1
            
            # 更新批次进度条
            batch_bar.set_postfix(loss=f"{loss.item():.6f}")
        
        avg_loss = epoch_loss / len(train_loader)
        train_losses.append(avg_loss)
        
        # 更新主进度条
        progress_bar.set_postfix(epoch_loss=f"{avg_loss:.6f}")
        
        # 打印每个epoch的损失
        if (epoch + 1) % Config.PRINT_EVERY == 0:
            elapsed = time.time() - start_time
            time_per_epoch = elapsed / (epoch + 1)
            remaining = time_per_epoch * (Config.NUM_EPOCHS - epoch - 1)
            
            print(f"Epoch [{epoch+1}/{Config.NUM_EPOCHS}] - Loss: {avg_loss:.6f} | "
                  f"耗时: {elapsed:.1f}s | 剩余: {remaining:.1f}s")
    
    training_time = time.time() - start_time
    print(f"\n模型训练完成! 总耗时: {training_time:.1f}秒")
    
    # 7. 测试模型：循环预测第15-30天的电价
    print(f"\n开始预测第{Config.TEST_START_DAY}到{Config.TEST_START_DAY + test_days - 1}天的电价...")
    model.eval()
    
    # 用于存储所有预测结果和真实值
    all_predictions = []
    all_actuals = []
    
    # 初始化预测起始点（第14天的结尾）
    current_idx = (Config.TEST_START_DAY - 1) * 96 - Config.SEQ_LEN
    current_input = scaled_prices[current_idx:current_idx + Config.SEQ_LEN]
    
    # 循环预测每一天
    for day in range(test_days):
        # 用于存储当前天的预测结果
        daily_predictions = []
        daily_actuals = []
        
        # 预测当前天的96个点（每次预测24个点）
        segments_per_day = 96 // Config.PRED_LEN
        
        # 使用进度条显示每天的预测进度
        day_progress = tqdm(range(segments_per_day), desc=f"预测第{Config.TEST_START_DAY + day}天")
        
        for segment in day_progress:
            # 准备输入数据
            input_seq = torch.FloatTensor(current_input).unsqueeze(0).to(Config.DEVICE)
            
            # 预测
            with torch.no_grad():
                pred = model(input_seq)
            
            # 转换回原始尺度
            pred = pred.cpu().numpy().flatten()
            pred = scaler.inverse_transform(pred.reshape(-1, 1)).flatten()
            
            # 获取实际值
            actual_start = test_start_idx + day * 96 + segment * Config.PRED_LEN
            actual_end = actual_start + Config.PRED_LEN
            
            if actual_end > len(prices):
                break
                
            actual = prices[actual_start:actual_end].flatten()
            
            # 存储预测和实际值
            daily_predictions.extend(pred)
            daily_actuals.extend(actual)
            
            # 更新当前输入序列：移除开头的Config.PRED_LEN个点，添加新预测的值
            if segment < segments_per_day - 1:
                # 移除最旧的Config.PRED_LEN个点
                current_input = np.concatenate([
                    current_input[Config.PRED_LEN:], 
                    # 使用真实值更新
                    scaled_prices[actual_start:actual_end]
                ])
        
        # 存储当前天的结果
        all_predictions.append(daily_predictions)
        all_actuals.append(daily_actuals)
        
        print(f"第{Config.TEST_START_DAY + day}天预测完成")
        
        # 准备下一天的输入序列：使用当天的最后Config.SEQ_LEN个点
        next_day_start = test_start_idx + day * 96 + (96 - Config.SEQ_LEN)
        next_day_end = next_day_start + Config.SEQ_LEN
        
        if next_day_end > len(scaled_prices):
            break
            
        current_input = scaled_prices[next_day_start:next_day_end]
    
    # 8. 计算评估指标并可视化结果
    flat_predictions = [p for day_predictions in all_predictions for p in day_predictions]
    flat_actuals = [a for day_actuals in all_actuals for a in day_actuals]
    
    rmse = np.sqrt(mean_squared_error(flat_actuals, flat_predictions))
    mae = mean_absolute_error(flat_actuals, flat_predictions)
    mean_price = np.mean(flat_actuals)
    relative_rmse = rmse / mean_price * 100 if mean_price != 0 else 0
    
    print("\n预测结果评估:")
    print(f"总RMSE: {rmse:.4f}")
    print(f"总MAE: {mae:.4f}")
    print(f"与平均电价相比: {relative_rmse:.2f}%")
    
    # 绘制训练损失曲线
    plt.figure(figsize=(10, 6))
    plt.plot(train_losses, label='Training Loss')
    plt.title('Model Training Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)
    plt.savefig('training_loss.png', dpi=300)
    
    # 绘制预测对比图
    plt.figure(figsize=(15, 8))
    
    # 为每一天创建子图
    num_days = len(all_predictions)
    if num_days > 0:
        for i in range(num_days):
            day_num = Config.TEST_START_DAY + i
            plt.subplot(4, 4, i+1)
            plt.plot(all_actuals[i], label='Actual', color='blue', alpha=0.7)
            plt.plot(all_predictions[i], label='Predicted', color='red', linestyle='--', alpha=0.7)
            plt.title(f'Day {day_num}')
            plt.grid(True)
            plt.legend()
        
        plt.tight_layout()
        plt.savefig('daily_predictions.png', dpi=300)
    
    # 绘制综合对比图
    plt.figure(figsize=(16, 8))
    all_days_actuals = np.concatenate(all_actuals)
    all_days_predictions = np.concatenate(all_predictions)
    
    plt.plot(all_days_actuals, label='Actual Prices', linewidth=1.5)
    plt.plot(all_days_predictions, label='Predicted Prices', linestyle='--', linewidth=1.5)
    
    plt.title('Electricity Price Prediction vs Actual')
    plt.xlabel('Time (15-min intervals)')
    plt.ylabel('Price')
    plt.legend()
    plt.grid(True)
    plt.savefig('overall_prediction_comparison.png', dpi=300)
    
    # 保存预测结果到CSV
    results = []
    for i in range(len(all_predictions)):
        for j in range(len(all_predictions[i])):
            hour = j // 4
            minute = (j % 4) * 15
            time_str = f"{hour:02d}:{minute:02d}"
            
            results.append({
                'Day': Config.TEST_START_DAY + i,
                'Time': time_str,
                'Actual': all_actuals[i][j],
                'Predicted': all_predictions[i][j]
            })
    
    results_df = pd.DataFrame(results)
    results_df.to_csv('prediction_results.csv', index=False)
    
    print("预测结果已保存到 prediction_results.csv")
    print("可视化结果已保存为图像文件")

if __name__ == "__main__":
    main()

使用设备: cpu
序列长度: 672 (相当于 7 天)
预测长度: 12 个时间点 (相当于 3.0 小时)
加载数据...
数据加载成功，共 28416 行
使用列 'price' 作为电价数据，共 28416 个数据点
总天数: 296
训练集大小: 1344 个点
测试集天数: 16 (第 15 到 30 天)
创建了 661 个训练样本

开始训练模型...


训练进度:   1%|          | 1/100 [00:05<09:03,  5.49s/epoch, epoch_loss=0.160309]

Epoch [1/100] - Loss: 0.160309 | 耗时: 5.5s | 剩余: 543.9s


训练进度:   2%|▏         | 2/100 [00:10<08:49,  5.40s/epoch, epoch_loss=0.138366]

Epoch [2/100] - Loss: 0.138366 | 耗时: 10.8s | 剩余: 530.7s


...

Epoch [3/100] - Loss: 0.112519 | 耗时: 16.2s | 剩余: 523.2s


训练进度:   4%|▍         | 4/100 [00:22<09:09,  5.72s/epoch, epoch_loss=0.086892]

Epoch [4/100] - Loss: 0.086892 | 耗时: 22.4s | 剩余: 538.2s


训练进度:   4%|▍         | 4/100 [00:23<09:27,  5.91s/epoch, epoch_loss=0.086892]


KeyboardInterrupt: 